# 04 — Stratified Group Split 80/10/10 · Banana Ripeness Detection (YOLO26s)

**หน้าที่ของ notebook นี้ (MASTER PROMPT ข้อ 4 + ข้อ 7 Paper 1)**
1. รวมภาพ "ซีนเดียวกัน" (scene_group จาก pHash ใน nb03 + ชื่อไฟล์) เป็น **split_group** — ภาพกลุ่มเดียวกันต้องอยู่ split เดียวกัน (กัน data leakage)
2. **Stratified Group Split 80/10/10** ด้วย `StratifiedGroupKFold` — strata = source × คลาสหลักของภาพ (ทุก split มีทั้ง Mendeley/Roboflow และทุกคลาสในสัดส่วนเดิม)
3. **Leakage check แบบ assert**: ไม่มีกลุ่มข้าม split, ไม่มีคู่ pHash ≤ 2 ข้าม split
4. สร้าง `datasets/banana/{train,valid,test}/{images,labels}` + `data.yaml` และตรวจด้วย `ultralytics.check_det_dataset`
5. **Oversampling list สำหรับ Improved experiment**: `train_oversampled.txt` + `data_oversampled.yaml` (ทำซ้ำ path ภาพ train ตาม ratio จาก nb03)
6. **Occlusion Test Subset** (Paper 1): ภาพ test ที่มี bbox ซ้อนทับกัน (IoU > 0) → `test_occlusion.txt` + `data_occlusion.yaml`
7. `reports/manifest_split.csv`, `reports/04_split_summary.json`, กราฟ `figures/04_*.png`, อัปเดต `configs/project.yaml`

> kernel: **ML Environment (ds69) + YOLO26** · ต้องรัน nb01–03 ก่อน (ใช้ `data/processed/` + `reports/manifest_clean.csv`)


In [ ]:
# === CELL 1 === โหลด config + manifest_clean + สร้าง split_group (กลุ่มภาพที่ห้ามแยกกัน)
# ทำอะไร : อ่าน configs/project.yaml, reports/manifest_clean.csv (จาก nb03) → union-find รวม scene_group (pHash ≤ 8) กับ group_id (ชื่อไฟล์)
# ทำไม   : split ต้องทำระดับ "กลุ่ม" ไม่ใช่ระดับภาพ — ภาพ burst/ซีนเดียวกันที่ไปอยู่ทั้ง train และ test ทำให้ mAP สูงเกินจริง (spec ข้อ 4)
import os, sys, re, json, shutil, random, importlib, math
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
import yaml
import numpy as np
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold

PROJECT_ROOT = Path.cwd().resolve()
CFG_PATH = PROJECT_ROOT / "configs" / "project.yaml"
assert CFG_PATH.exists(), f"ไม่พบ {CFG_PATH} → รัน 01_setup_env.ipynb ก่อน"
with open(CFG_PATH, encoding="utf-8") as f:
    CONFIG = yaml.safe_load(f)
DIRS = {k: PROJECT_ROOT / v for k, v in CONFIG["dirs"].items()}
SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
CLASSES = {int(k): v for k, v in CONFIG["classes"].items()}
NAMES = list(CLASSES.values())
NC = len(CLASSES)
sys.path.insert(0, str(DIRS["src"]))
import viz
importlib.reload(viz)
viz.apply_style()
CLASS_RGB = viz.class_colors(CONFIG)

IM_IN, LB_IN = DIRS["data_processed"] / "images", DIRS["data_processed"] / "labels"
MAN = DIRS["reports"] / "manifest_clean.csv"
assert MAN.exists(), "ไม่พบ reports/manifest_clean.csv → รัน 03_data_cleaning_eda.ipynb ก่อน"
df = pd.read_csv(MAN, dtype={"phash": str})
n_files = sum(1 for p in IM_IN.iterdir() if p.is_file())
assert len(df) == n_files, f"manifest_clean {len(df)} แถว แต่ data/processed มี {n_files} ไฟล์ → รัน nb03 ใหม่"
assert "oversampling_plan" in CONFIG, "ไม่พบ oversampling_plan ใน project.yaml → รัน nb03 ให้จบ"

# --- ค่าตั้งของ nb04 (รวมไว้ที่เดียว) ---
NB04 = {
    "n_folds": 10,                     # 10 fold → 1 fold = 10% → test 1 fold, valid 1 fold, train 8 fold = 80/10/10
    "strata": "source_x_class",        # stratify ตาม source × คลาสหลักของภาพ (8 ชั้น)
    "dup_max_dist": int(CONFIG.get("cleaning_thresholds", {}).get("dup_hamming", {}).get("exact", 2)),   # เกณฑ์ซ้ำเดียวกับ nb03
    "occlusion_iou": 0.0,              # occlusion subset = ภาพ test ที่มี bbox อย่างน้อย 1 คู่ซ้อนกัน (IoU > 0) — ตกลง 2026-09-20
    "occlusion_report_iou": 0.10,      # รายงานเพิ่มว่ามีกี่ภาพที่ซ้อนกันชัด (IoU > 0.1)
    "oversample_plan": {k: float(v) for k, v in CONFIG["oversampling_plan"].items()},   # จาก nb03 (train เท่านั้น)
}
DS = DIRS["datasets"]                  # datasets/banana
RUN = {"notebook": "04_split_dataset", "started_at": datetime.now().isoformat(timespec="seconds"), "settings": NB04}

# --- split_group = union(scene_group, group_id) ---
parent = {}
def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra
for r in df.itertuples():
    union(f"s:{r.scene_group}", f"g:{r.group_id}")
df["split_group"] = [find(f"s:{g}") for g in df["scene_group"]]
gsize = df.groupby("split_group").size()
df["group_size"] = df["split_group"].map(gsize)
bins = pd.cut(gsize, [0, 1, 5, 20, 10**6], labels=["1", "2–5", "6–20", ">20"])
print(f"[OK] manifest_clean: {len(df):,} ภาพ | split_group {gsize.size:,} กลุ่ม | กลุ่มที่มี >1 ภาพ {(gsize > 1).sum():,} | ใหญ่สุด {gsize.max()} ภาพ")
print("     ขนาดกลุ่ม:", {str(k): int(v) for k, v in bins.value_counts().sort_index().items()})
big = gsize.sort_values(ascending=False).head(3)
for g, n in big.items():
    sub = df[df.split_group == g]
    print(f"     กลุ่มใหญ่: {n} ภาพ | source { {k: int(v) for k, v in sub.source.value_counts().items()} } | dominant { {k: int(v) for k, v in sub.dominant_name.value_counts().items()} }")
RUN["groups"] = {"n_groups": int(gsize.size), "multi_image_groups": int((gsize > 1).sum()), "max_group_size": int(gsize.max())}


In [ ]:
# === CELL 2 === Stratified Group Split 80/10/10 (seed 42) + ตาราง/กราฟการกระจายต่อ split
# ทำอะไร : StratifiedGroupKFold 10 fold บน strata = source×dominant_class, groups = split_group → เลือก 2 fold ที่ขนาดใกล้ 10% ที่สุดเป็น test/valid
# ทำไม   : spec ข้อ 4 บังคับ stratified ตามคลาสหลักของภาพ + กัน leakage; แยก strata ตาม source ด้วยเพื่อให้ test มีทั้งภาพสตูดิโอ (Mendeley) และภาพจริง (Roboflow)
y = (df["source"] + "_" + df["dominant_name"]).to_numpy()
groups = df["split_group"].to_numpy()
sgkf = StratifiedGroupKFold(n_splits=NB04["n_folds"], shuffle=True, random_state=SEED)
fold = np.full(len(df), -1)
for k, (_, idx) in enumerate(sgkf.split(df, y, groups)):
    fold[idx] = k
assert (fold >= 0).all(), "บางภาพไม่ได้รับ fold"
target = len(df) / NB04["n_folds"]
sizes = pd.Series(fold).value_counts().sort_index()
order = (sizes - target).abs().sort_values(kind="stable").index.tolist()   # fold ที่ขนาดใกล้ 10% ที่สุดก่อน
test_fold, valid_fold = int(order[0]), int(order[1])
df["split"] = np.where(fold == test_fold, "test", np.where(fold == valid_fold, "valid", "train"))
print(f"[OK] fold sizes: {sizes.to_dict()} | target/fold = {target:.0f} → test = fold {test_fold}, valid = fold {valid_fold}, train = ที่เหลือ 8 fold")

SPLITS = ["train", "valid", "test"]
n_split = df["split"].value_counts().reindex(SPLITS)
print("\n— ภาพต่อ split —")
print(pd.DataFrame({"images": n_split, "share": (n_split / len(df) * 100).round(1).astype(str) + "%"}).T)
print("\n— ภาพต่อ split × source —")
print(pd.crosstab(df["split"], df["source"], margins=True).reindex([*SPLITS, "All"]))
print("\n— ภาพต่อ split × คลาสหลัก (image-level) —")
img_tab = pd.crosstab(df["split"], df["dominant_name"]).reindex(index=SPLITS, columns=NAMES)
print(img_tab.assign(All=img_tab.sum(axis=1)))
print("   สัดส่วนต่อคลาสในแต่ละ split (%):")
print((img_tab.div(img_tab.sum(axis=1), axis=0) * 100).round(1))
inst_cols = [f"n_{n}" for n in NAMES]
inst_tab = df.groupby("split")[inst_cols].sum().reindex(SPLITS)
inst_tab.columns = NAMES
print("\n— instances (bbox) ต่อ split × คลาส —")
print(inst_tab.assign(All=inst_tab.sum(axis=1)))

# assert เชิงคุณภาพ: ทุก split มีครบทุกคลาส และ valid/test อยู่ในช่วง 8–12%
assert (img_tab > 0).all().all(), "มี split ที่ไม่มีบางคลาสเลย"
for s_ in ("valid", "test"):
    share = n_split[s_] / len(df)
    assert 0.08 <= share <= 0.12, f"{s_} = {share:.1%} หลุดช่วง 8–12% (กลุ่มใหญ่ผิดปกติ?)"
RUN["split"] = {"test_fold": test_fold, "valid_fold": valid_fold, "images": n_split.to_dict(),
                "images_by_class": {s_: img_tab.loc[s_].to_dict() for s_ in SPLITS},
                "instances_by_class": {s_: inst_tab.loc[s_].to_dict() for s_ in SPLITS}}

# --- กราฟ: สัดส่วนคลาสต่อ split (100% stacked) เพื่อให้เห็นว่า stratify ได้สม่ำเสมอ + จำนวนรวมกำกับบนแท่ง ---
fig, axes = plt.subplots(1, 2, figsize=(11, 4.4))
for ax, tab, title in ((axes[0], img_tab, "Images per split — class share (dominant class)"), (axes[1], inst_tab, "Instances per split — class share")):
    pct = tab.div(tab.sum(axis=1), axis=0) * 100
    bottom = np.zeros(len(SPLITS))
    for c, name in CLASSES.items():
        vals = pct[name].to_numpy()
        ax.bar(SPLITS, vals, bottom=bottom, color=CLASS_RGB[c], label=name, width=0.6, edgecolor="white", linewidth=1)
        for i, v in enumerate(vals):
            if v >= 6:
                ax.text(i, bottom[i] + v / 2, f"{v:.0f}%", ha="center", va="center", fontsize=8, color="white")
        bottom += vals
    for i, s_ in enumerate(SPLITS):
        ax.text(i, 101, f"n = {int(tab.loc[s_].sum()):,}", ha="center", va="bottom", fontsize=9, color=viz.TEXT["secondary"])
    ax.set_ylim(0, 112)
    ax.set_ylabel("share (%)")
    ax.set_title(title, loc="left", fontsize=11)
handles, labels = axes[0].get_legend_handles_labels()
fig.suptitle(f"Stratified group split 80/10/10 — {len(df):,} images, {gsize.size:,} scene groups (seed {SEED})", x=0.01, ha="left", fontsize=12)
fig.tight_layout(rect=(0, 0.07, 1, 0.95))
fig.legend(handles, labels, loc="lower center", ncol=4, bbox_to_anchor=(0.5, 0.0))
out = viz.save_fig(fig, DIRS["figures"], "04_split_distribution.png")
plt.show()
plt.close(fig)
print(f"[OK] {out.relative_to(PROJECT_ROOT)}")


In [ ]:
# === CELL 3 === Leakage check (assert): กลุ่มไม่ข้าม split + ไม่มีคู่ pHash ≤ 2 ข้าม split + คู่ stage_pair อยู่ด้วยกัน
# ทำอะไร : ตรวจ 3 อย่างด้วยข้อมูลจริงหลัง split (ไม่เชื่อแค่ทฤษฎีของ GroupKFold)
# ทำไม   : ถ้ารั่วแม้คู่เดียว ตัวเลข mAP บน test จะเชื่อถือไม่ได้ — spec ข้อ 4 ให้กันไว้ตั้งแต่ตอน split
# (1) ทุก split_group อยู่ split เดียว
per_group = df.groupby("split_group")["split"].nunique()
assert per_group.max() == 1, f"พบกลุ่มที่กระจายหลาย split: {int((per_group > 1).sum())} กลุ่ม"

# (2) ไม่มีคู่ภาพที่ pHash ใกล้กัน (≤ dup_max_dist) ข้าม split — คำนวณระยะจริงทุกคู่ระหว่าง (valid ∪ test) กับ split อื่น
hashes = np.array([int(h, 16) for h in df["phash"]], dtype=np.uint64)
POPCOUNT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint8)
split_arr = df["split"].to_numpy()
D = NB04["dup_max_dist"]
leaks, min_cross = [], 99
eval_idx = np.where(split_arr != "train")[0]
for i in tqdm(eval_idx, desc="cross-split hash check"):
    d = POPCOUNT[np.bitwise_xor(hashes[i], hashes).view(np.uint8)].reshape(len(hashes), 8).sum(axis=1)
    other = split_arr != split_arr[i]
    dmin = int(d[other].min())
    min_cross = min(min_cross, dmin)
    if dmin <= D:
        j = int(np.where(other)[0][d[other].argmin()])
        leaks.append((df.file.iloc[i], split_arr[i], df.file.iloc[j], split_arr[j], dmin))
assert not leaks, f"พบภาพใกล้เคียง (pHash ≤ {D}) ข้าม split {len(leaks)} คู่ เช่น {leaks[:3]}"

# (3) ภาพที่ติด stage_pair_flag (ภาพเดิม label ต่าง) อยู่ split เดียวกันกับคู่ของมัน (อยู่ scene_group เดียวกัน → ผ่านข้อ 1 อยู่แล้ว แต่รายงานให้เห็น)
n_stage = int(df["stage_pair_flag"].sum()) if "stage_pair_flag" in df.columns else 0
print(f"[OK] leakage check ผ่าน: {gsize.size:,} กลุ่มอยู่ split เดียวทั้งหมด | ระยะ pHash ใกล้สุดข้าม split = {min_cross} (> {D}) | "
      f"stage_pair images {n_stage} อยู่กับคู่ของตัวเอง")
RUN["leakage_check"] = {"groups_cross_split": 0, "min_cross_split_hamming": int(min_cross), "dup_threshold": D, "cross_split_near_pairs": 0}


In [ ]:
# === CELL 4 === สร้าง datasets/banana/{train,valid,test}/{images,labels} + data.yaml + ตรวจด้วย ultralytics
# ทำอะไร : copy ไฟล์จาก data/processed ตาม split (เริ่มใหม่ทุกครั้ง idempotent) → เขียน data.yaml (path/train/val/test/nc/names)
#          → check_det_dataset ของ ultralytics ยืนยันว่าโครงสร้างถูกต้องก่อนถึง nb05
# ทำไม   : ultralytics หา label จากการแทน /images/ ด้วย /labels/ ใน path → โครงสร้างต้องเป๊ะ ไม่งั้นเทรนแล้ว "no labels found" เงียบ ๆ
for s_ in SPLITS:
    for sub in ("images", "labels"):
        d = DS / s_ / sub
        shutil.rmtree(d, ignore_errors=True)
        d.mkdir(parents=True, exist_ok=True)
for r in tqdm(df.itertuples(), total=len(df), desc="copy to datasets/banana"):
    shutil.copy2(IM_IN / r.file, DS / r.split / "images" / r.file)
    shutil.copy2(LB_IN / r.label, DS / r.split / "labels" / r.label)
counts = {s_: sum(1 for _ in (DS / s_ / "images").iterdir()) for s_ in SPLITS}
assert counts == n_split.to_dict(), f"จำนวนไฟล์ที่ copy {counts} ไม่ตรง split {n_split.to_dict()}"

data_yaml = {
    "path": str(DS), "train": "train/images", "val": "valid/images", "test": "test/images",
    "nc": NC, "names": {k: v for k, v in CLASSES.items()},
}
DATA_YAML = DS / "data.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)

from ultralytics.data.utils import check_det_dataset
info = check_det_dataset(str(DATA_YAML))
assert int(info["nc"]) == NC and list(info["names"].values()) == NAMES, f"data.yaml ไม่ตรง taxonomy: {info.get('names')}"
print(f"[OK] {DATA_YAML.relative_to(PROJECT_ROOT)}: " + " | ".join(f"{s_} {counts[s_]:,}" for s_ in SPLITS) + f" | nc={info['nc']} names={list(info['names'].values())}")
size_gb = sum(p.stat().st_size for s_ in SPLITS for p in (DS / s_ / "images").iterdir()) / 1e9
print(f"[OK] ultralytics check_det_dataset ผ่าน | ขนาดรวม {size_gb:.2f} GB")
RUN["dataset"] = {"data_yaml": str(DATA_YAML.relative_to(PROJECT_ROOT)), "counts": counts, "size_gb": round(size_gb, 3)}


In [ ]:
# === CELL 5 === Oversampling list สำหรับ Improved experiment: train_oversampled.txt + data_oversampled.yaml
# ทำอะไร : ทำซ้ำ path ของภาพ train ตาม oversampling_plan (คลาสหลักของภาพ) จาก nb03 — เศษทศนิยมสุ่มด้วย seed → ได้ ratio เฉลี่ยตรงแผน
#          เขียน data_oversampled.yaml ที่ชี้ train ไปยังไฟล์ txt (ultralytics รับ list ของ path และอนุญาตให้ซ้ำได้)
# ทำไม   : ultralytics detect ไม่มี class_weights → วิธีมาตรฐานคือทำซ้ำภาพคลาสน้อยใน train; valid/test ห้ามแตะ (ประเมินบนการกระจายจริง)
plan = NB04["oversample_plan"]
rng = random.Random(SEED)
train_df = df[df.split == "train"]
lines, reps_by_class = [], Counter()
for r in train_df.itertuples():
    ratio = plan.get(r.dominant_name, 1.0)
    reps = int(ratio) + (1 if rng.random() < (ratio - int(ratio)) else 0)
    reps = max(1, reps)
    lines += [str((DS / "train" / "images" / r.file).resolve())] * reps
    reps_by_class[r.dominant_name] += reps
TRAIN_OS = DS / "train_oversampled.txt"
TRAIN_OS.write_text("\n".join(lines) + "\n", encoding="utf-8")
os_yaml = {**data_yaml, "train": "train_oversampled.txt"}
DATA_OS_YAML = DS / "data_oversampled.yaml"
with open(DATA_OS_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(os_yaml, f, allow_unicode=True, sort_keys=False)
info_os = check_det_dataset(str(DATA_OS_YAML))

# ผลต่อ instance-level ใน train: นับ bbox × จำนวนครั้งที่ภาพถูกทำซ้ำ
rep_count = Counter(lines)
inst_before = train_df[inst_cols].sum().to_numpy()
inst_after = np.zeros(NC)
for r in train_df.itertuples():
    k = rep_count[str((DS / "train" / "images" / r.file).resolve())]
    inst_after += np.array([getattr(r, f"n_{n}") for n in NAMES]) * k
imb_before, imb_after = inst_before.max() / max(inst_before.min(), 1), inst_after.max() / max(inst_after.min(), 1)
print(f"[OK] {TRAIN_OS.relative_to(PROJECT_ROOT)}: train {len(train_df):,} ภาพ → {len(lines):,} รายการ (ratio ตามคลาสหลัก: " +
      ", ".join(f"{n} ×{plan[n]:.2f}" for n in NAMES) + ")")
print("     instances ใน train ก่อน → หลัง oversampling: " + ", ".join(f"{n} {int(b):,}→{int(a):,}" for n, b, a in zip(NAMES, inst_before, inst_after)))
print(f"     imbalance ratio (max/min) {imb_before:.2f} → {imb_after:.2f} | valid/test ไม่เปลี่ยน")
print(f"[OK] {DATA_OS_YAML.relative_to(PROJECT_ROOT)} ผ่าน check_det_dataset (train = txt list, val/test เหมือน data.yaml)")

fig, ax = plt.subplots(figsize=(7.5, 3.8))
x = np.arange(NC)
b1 = ax.bar(x - 0.19, inst_before, width=0.36, color="#b9b8b1", label="train (as split)")
b2 = ax.bar(x + 0.19, inst_after, width=0.36, color=viz.SOURCE_COLORS["mendeley"], label="train (oversampled list)")
for bars in (b1, b2):
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{int(bar.get_height()):,}", ha="center", va="bottom", fontsize=8, color=viz.TEXT["secondary"])
ax.set_xticks(x, NAMES)
ax.set_title(f"Train-set instances before / after oversampling — imbalance {imb_before:.2f} → {imb_after:.2f}", loc="left", fontsize=11)
ax.margins(y=0.15)
ax.legend(loc="upper left")
fig.tight_layout()
out = viz.save_fig(fig, DIRS["figures"], "04_oversampling_effect.png")
plt.show()
plt.close(fig)
RUN["oversampling"] = {"train_images": int(len(train_df)), "train_entries": len(lines), "plan": plan,
                       "instances_before": dict(zip(NAMES, inst_before.astype(int).tolist())),
                       "instances_after": dict(zip(NAMES, inst_after.astype(int).tolist())),
                       "imbalance_before": round(float(imb_before), 3), "imbalance_after": round(float(imb_after), 3)}


In [ ]:
# === CELL 6 === Occlusion Test Subset (Paper 1): ภาพ test ที่มี bbox ซ้อนทับกัน (IoU > 0) → test_occlusion.txt + data_occlusion.yaml
# ทำอะไร : คำนวณ IoU ทุกคู่ bbox ในแต่ละภาพ test → ภาพที่มีคู่ซ้อนกัน = occlusion subset, ที่เหลือ = clear subset (ไว้เทียบ)
#          + รายงาน multi-object (≥2 ผล) และจำนวนที่ซ้อนชัด (IoU > 0.1) + grid ตัวอย่าง
# ทำไม   : spec ข้อ 7 Paper 1 ให้รายงานผลบนภาพกล้วยหลายผล/ซ้อนทับแยกจาก test ปกติ และใช้เทียบ NMS vs NMS-free ใน nb06
def parse_label(path: Path):
    rows = []
    for ln in Path(path).read_text(encoding="utf-8").splitlines():
        p = ln.split()
        if len(p) >= 5:
            rows.append((int(float(p[0])), float(p[1]), float(p[2]), float(p[3]), float(p[4])))
    return rows

def iou_xywh(a, b):
    ax1, ay1, ax2, ay2 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
    bx1, by1, bx2, by2 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    iw, ih = max(0.0, min(ax2, bx2) - max(ax1, bx1)), max(0.0, min(ay2, by2) - max(ay1, by1))
    inter = iw * ih
    union = a[2] * a[3] + b[2] * b[3] - inter
    return inter / union if union > 0 else 0.0

test_df = df[df.split == "test"].copy()
max_iou, n_overlap_pairs = {}, {}
for r in test_df.itertuples():
    boxes = parse_label(DS / "test" / "labels" / r.label)
    ious = [iou_xywh(boxes[i][1:], boxes[j][1:]) for i in range(len(boxes)) for j in range(i + 1, len(boxes))]
    max_iou[r.file] = max(ious) if ious else 0.0
    n_overlap_pairs[r.file] = sum(1 for v in ious if v > NB04["occlusion_iou"])
test_df["max_iou"] = test_df["file"].map(max_iou)
test_df["overlap_pairs"] = test_df["file"].map(n_overlap_pairs)
test_df["occlusion"] = test_df["max_iou"] > NB04["occlusion_iou"]
occ = test_df[test_df.occlusion]
n_multi = int((test_df.n_boxes >= 2).sum())
n_clear_iou = int((test_df.max_iou > NB04["occlusion_report_iou"]).sum())

TEST_OCC = DS / "test_occlusion.txt"
TEST_CLEAR = DS / "test_clear.txt"
TEST_OCC.write_text("\n".join(str((DS / "test" / "images" / f).resolve()) for f in occ["file"]) + "\n", encoding="utf-8")
TEST_CLEAR.write_text("\n".join(str((DS / "test" / "images" / f).resolve()) for f in test_df[~test_df.occlusion]["file"]) + "\n", encoding="utf-8")
DATA_OCC_YAML = DS / "data_occlusion.yaml"
with open(DATA_OCC_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump({**data_yaml, "val": "test_occlusion.txt", "test": "test_occlusion.txt",
                    "note": "occlusion subset of the test split (>=1 pair of boxes with IoU > 0) — use val= for model.val()"}, f, allow_unicode=True, sort_keys=False)
DATA_CLEAR_YAML = DS / "data_clear.yaml"
with open(DATA_CLEAR_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump({**data_yaml, "val": "test_clear.txt", "test": "test_clear.txt",
                    "note": "complement of the occlusion subset (no overlapping boxes)"}, f, allow_unicode=True, sort_keys=False)
check_det_dataset(str(DATA_OCC_YAML))
print(f"[OK] test {len(test_df):,} ภาพ | multi-object (≥2 bbox) {n_multi:,} | occlusion (IoU > {NB04['occlusion_iou']:g}) {len(occ):,} ภาพ "
      f"({len(occ) / len(test_df) * 100:.1f}%) | ซ้อนชัด (IoU > {NB04['occlusion_report_iou']}) {n_clear_iou:,} | clear {len(test_df) - len(occ):,}")
print(f"     occlusion subset ต่อ source: { {k: int(v) for k, v in occ.source.value_counts().items()} } | ต่อคลาสหลัก: { {k: int(v) for k, v in occ.dominant_name.value_counts().items()} }")
print(f"     bbox ต่อภาพใน occlusion subset: mean {occ.n_boxes.mean():.2f} (max {occ.n_boxes.max()}) | max IoU ต่อภาพ: median {occ.max_iou.median():.3f}")
print(f"[OK] {TEST_OCC.relative_to(PROJECT_ROOT)}, {TEST_CLEAR.relative_to(PROJECT_ROOT)}, {DATA_OCC_YAML.relative_to(PROJECT_ROOT)}, {DATA_CLEAR_YAML.relative_to(PROJECT_ROOT)}")
RUN["occlusion_subset"] = {"test_images": int(len(test_df)), "multi_object": n_multi, "occlusion": int(len(occ)),
                           "occlusion_iou_gt_0_1": n_clear_iou, "clear": int(len(test_df) - len(occ)),
                           "by_source": {k: int(v) for k, v in occ.source.value_counts().items()}}

# --- grid ตัวอย่าง occlusion (max IoU สูงสุดก่อน) ---
from PIL import Image, ImageDraw
CLASS_RGB255 = {k: tuple(int(c * 255) for c in v) for k, v in CLASS_RGB.items()}
ex = occ.sort_values("max_iou", ascending=False).head(8)
if len(ex):
    fig, axes = plt.subplots(2, 4, figsize=(11, 5.8))
    for ax, r in zip(axes.reshape(-1), ex.itertuples()):
        with Image.open(DS / "test" / "images" / r.file) as im:
            im = im.convert("RGB")
            s_ = 320 / max(im.size)
            im = im.resize((max(1, round(im.width * s_)), max(1, round(im.height * s_))))
            d = ImageDraw.Draw(im)
            W, H = im.size
            for c, x, y, w, h in parse_label(DS / "test" / "labels" / r.label):
                d.rectangle([(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H], outline=CLASS_RGB255[c], width=3)
            ax.imshow(im)
        ax.axis("off")
        ax.set_title(f"{r.source[:3]} · {r.n_boxes} boxes · max IoU {r.max_iou:.2f}", fontsize=8, loc="left")
    for ax in axes.reshape(-1)[len(ex):]:
        ax.axis("off")
    fig.suptitle(f"Occlusion test subset — {len(occ)} of {len(test_df)} test images have overlapping boxes (IoU > 0)", x=0.01, ha="left", fontsize=11)
    fig.tight_layout()
    out = viz.save_fig(fig, DIRS["figures"], "04_occlusion_examples.png")
    plt.show()
    plt.close(fig)
    print(f"[OK] {out.relative_to(PROJECT_ROOT)}")
df = df.merge(test_df[["file", "max_iou", "occlusion"]], on="file", how="left")


In [ ]:
# === CELL 7 === บันทึก manifest_split.csv + 04_split_summary.json + อัปเดต project.yaml + สรุป
# ทำอะไร : เก็บ split ของทุกภาพ (nb05/06 ใช้อ้างอิง) และบันทึก path ของ data yaml ทั้ง 4 ไฟล์ลง config กลาง
# ทำไม   : nb05 ต้องรู้ว่า Baseline ใช้ data.yaml, Improved ใช้ data_oversampled.yaml; nb06 ต้องรู้ occlusion subset — ทุกอย่างอ่านจาก project.yaml
mcols = ["file", "label", "source", "split", "split_group", "group_size", "dominant_class", "dominant_name", "n_boxes", *inst_cols,
         "width", "height", "max_iou", "occlusion"]
manifest_split = DIRS["reports"] / "manifest_split.csv"
df[mcols].to_csv(manifest_split, index=False, encoding="utf-8-sig")

CONFIG["datasets"]["actual_split"] = {**{s_: int(n_split[s_]) for s_ in SPLITS}, "total": int(len(df)),
                                      "train_oversampled_entries": len(lines), "test_occlusion": int(len(occ)), "test_clear": int(len(test_df) - len(occ)),
                                      "scene_groups": int(gsize.size), "seed": SEED, "updated_by": "04_split_dataset.ipynb"}
CONFIG["data_yaml"] = {
    "baseline": str(DATA_YAML.relative_to(PROJECT_ROOT)),          # Baseline experiment (nb05)
    "improved": str(DATA_OS_YAML.relative_to(PROJECT_ROOT)),       # Improved experiment: oversampled train (nb05)
    "test_occlusion": str(DATA_OCC_YAML.relative_to(PROJECT_ROOT)),  # occlusion subset (nb06)
    "test_clear": str(DATA_CLEAR_YAML.relative_to(PROJECT_ROOT)),
    "train_list": str(TRAIN_OS.relative_to(PROJECT_ROOT)), "test_occlusion_list": str(TEST_OCC.relative_to(PROJECT_ROOT)),
}
with open(CFG_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(CONFIG, f, allow_unicode=True, sort_keys=False)
RUN["finished_at"] = datetime.now().isoformat(timespec="seconds")
with open(DIRS["reports"] / "04_split_summary.json", "w", encoding="utf-8") as f:
    json.dump(RUN, f, ensure_ascii=False, indent=2, default=str)

spec = CONFIG["datasets"]["expected_split"]
print(f"[OK] {manifest_split.relative_to(PROJECT_ROOT)} ({len(df):,} แถว) | reports/04_split_summary.json | project.yaml → datasets.actual_split + data_yaml")
print("\n— SPEC vs ACTUAL —")
for s_ in SPLITS:
    print(f"  {s_:<6}: spec {spec[s_]:,} → actual {int(n_split[s_]):,} ({n_split[s_] / len(df) * 100:.1f}%)")
print("\n=== SUMMARY · Notebook 04 ===")
print(f"  datasets/banana: train {int(n_split['train']):,} / valid {int(n_split['valid']):,} / test {int(n_split['test']):,} | scene groups {gsize.size:,} | leakage 0")
print(f"  Improved train list: {len(lines):,} รายการ (imbalance {imb_before:.2f} → {imb_after:.2f}) | occlusion subset: {len(occ):,} / {len(test_df):,} test images")
print(f"  ถัดไป: 05_train_yolo26s.ipynb (Baseline = data.yaml, Improved = data_oversampled.yaml)")


## ผลลัพธ์ที่ควรเห็น
- CELL 2: fold sizes ใกล้ 10% ทุก fold, ตารางสัดส่วนคลาสต่อ split ใกล้เคียงกัน (ต่างกันไม่เกิน ~2–3% ต่อคลาส), valid/test อยู่ในช่วง 8–12%
- CELL 3: `[OK] leakage check ผ่าน ... ระยะ pHash ใกล้สุดข้าม split = N (> 2)` — ถ้า assert ล้มเหลว **ห้ามไปต่อ**
- CELL 4: `ultralytics check_det_dataset ผ่าน` และจำนวนไฟล์ตรงกับตาราง
- CELL 5: instances ใน train หลัง oversampling ใกล้เคียงกันทุกคลาส (imbalance → ~1.0–1.3)
- CELL 6: จำนวน occlusion subset (คาดว่ามาจาก Roboflow เกือบทั้งหมด เพราะ Mendeley ภาพละ 1 ผล)
- ไฟล์: `datasets/banana/{data.yaml, data_oversampled.yaml, data_occlusion.yaml, data_clear.yaml, train_oversampled.txt, test_occlusion.txt}`

## ปัญหาที่อาจเจอ + วิธีแก้
| อาการ | สาเหตุ | วิธีแก้ |
|---|---|---|
| `valid = 7.x% หลุดช่วง 8–12%` | มีกลุ่มภาพซีนเดียวกันขนาดใหญ่ผิดปกติ | ดูบรรทัด "กลุ่มใหญ่" ใน CELL 1 — ถ้ากลุ่ม > 100 ภาพ ให้ลด `scene_max_dist` ใน nb03 จาก 8 → 6 แล้วรัน nb03–04 ใหม่ |
| `พบภาพใกล้เคียง ... ข้าม split` | scene_group ไม่ครอบคลุม | ส่ง output มา — จะปรับ union ให้รวมคู่นั้น |
| `check_det_dataset` เตือน "no labels" | copy ไม่ครบ | Run All ใหม่ (CELL 4 ล้างโฟลเดอร์ก่อน copy ทุกครั้ง) |
| `ImportError: StratifiedGroupKFold` | scikit-learn เก่ากว่า 1.0 | `pip install -U scikit-learn` ใน kernel นี้ |
